# Hate Speech Classification with LLMs

This notebook explores using Large Language Models (LLMs) for hate speech classification on the HateXplain dataset.
We will use a prompt engineering approach to classify text into "hatespeech", "normal", or "offensive".

> **Important:** Please ensure you are using a **T4 GPU** runtime in Google Colab for faster inference.
> Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [6]:
!pip install -q openai "datasets<3.0" "pyarrow<20" scikit-learn pandas python-dotenv tqdm zstd

In [10]:
import subprocess
import time
import os

# 1. Install zstd (dependency for Ollama installation)
print("Installing zstd...")
!apt-get update && apt-get install -y zstd

# 2. Install Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start Ollama server in the background
# We use Popen to run it asynchronously so it doesn't block the notebook
print("Starting Ollama server...")
process = subprocess.Popen("ollama serve", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, preexec_fn=os.setsid) # Use os.setsid to create a new session for the child process
time.sleep(10) # Give it a few seconds to initialize

# 4. Pull the model
# User requested ~4B model. 'gemma3' doesn't exist yet.
# 'phi3' is Microsoft's 3.8B model, which is excellent and fits the size request perfectly.
print("Pulling model...")
!ollama pull qwen3:1.7b

Installing zstd...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspe

In [11]:
import os
import json
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI Client for Local Ollama
# Ollama provides an OpenAI-compatible API at /v1
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # API key is required but ignored by Ollama
)

print("Client initialized for local Ollama.")

Client initialized for local Ollama.


In [12]:
# Data Processing Helpers (Reused from BERT Notebook)

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    if ann is None: return []
    if isinstance(ann, str):
        try: ann = json.loads(ann)
        except: return []
    if isinstance(ann, list):
        if len(ann) == 0: return []
        if isinstance(ann[0], dict): return ann
        out = []
        for x in ann:
            try: out.append({"label": int(x), "target": []})
            except: pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        n = len(labels)
        out = []
        for i in range(n):
            try: lab = int(labels[i])
            except: continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    if rats is None: return []
    if isinstance(rats, str):
        try: rats = json.loads(rats)
        except: return []
    if isinstance(rats, list):
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats: return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex: return str(ex["text"])
    if "cleaned_text" in ex: return str(ex["cleaned_text"])
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,
        "targets": targets,
    }

In [42]:
# Load and Prepare Data
print("Loading HateXplain dataset...")
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# Normalize
# Removed num_proc=2 to avoid pickling errors on Windows where helper functions aren't found in worker processes
ds = ds.map(normalize_example)

# Keep only necessary columns
to_keep = ["text_norm", "label_norm"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Filter empty
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

# Filter out 'offensive' examples
print("Filtering out 'offensive' examples...")
ds = ds.filter(lambda example: example["label_norm"] != NAME2ID["offensive"])

Loading HateXplain dataset...
Filtering out 'offensive' examples...


In [43]:
# Select Test Set
test_ds = ds["test"].select(range(100))
print(f"Test set size (subset of 100): {len(test_ds)}")

Test set size (subset of 100): 100


In [64]:
# Restart Ollama Server (Run this if you get Connection Errors)
import subprocess
import time

print("Stopping any existing Ollama processes...")
!pkill ollama
time.sleep(2)

print("Starting Ollama server...")
process = subprocess.Popen("ollama serve", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10) # Wait for server to start

print("Server restarted. Verifying...")
!ollama list

Stopping any existing Ollama processes...
Starting Ollama server...
Server restarted. Verifying...
NAME          ID              SIZE      MODIFIED       
gemma3:4b     a2af6cc3eb7f    3.3 GB    7 minutes ago     
qwen3:1.7b    8f68893c685c    1.4 GB    36 minutes ago    


In [65]:
# run to call other models
!ollama pull phi3

In [66]:
def classify_text(text, model="phi3"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull phi3`
    """

    system_prompt = (
        "You are an expert content moderator. "
        "Classify the following text into one of these two categories: 'hatespeech', 'normal'.\n\n"
        "Definitions:\n"
        "- 'hatespeech': Content that promotes violence, discrimination, or disparagement against protected groups (race, religion, gender, etc.). It aims to incite hatred.\n"
        "Provide ONLY the category name as your response, in lowercase.\n\n"
    )

    user_prompt = f"Text: {text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=10,
            timeout=60
        )
        prediction_content = response.choices[0].message.content
        prediction = prediction_content.strip().lower()

        if not prediction: # Treat empty response as an error
            print(f"Warning: Model returned empty prediction for text: '{text[:50]}...' (truncated)")
            print(f"Raw model response content: '{prediction_content}' (length: {len(prediction_content)})\n")
            return "empty_prediction_error"
        return prediction
    except Exception as e:
        print(f"Error during classification: {e}")
        return "api_error"

# Test the function
# Ensure you have the model running in Ollama
sample_text = test_ds[0]['text_norm']
print(f"Sample Text: {sample_text}")
print(f"Prediction: {classify_text(sample_text)}")

Sample Text: <user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children
Prediction: hatespeech


In [67]:
# Run Inference on Test Set
predictions = []
ground_truth = []
texts = []

print("Running inference...")
for example in tqdm(test_ds):
    text = example['text_norm']
    label_id = example['label_norm']

    true_label = ID2NAME[label_id]

    pred_label = classify_text(text)

    # Handle potential API errors or unexpected output
    if pred_label not in NAME2ID or pred_label == "normal": # Explicitly handle 'normal' if LLM still outputs it
        if "hate" in pred_label: pred_label = "hatespeech"
        # elif "offen" in pred_label: pred_label = "offensive"
        else: pred_label = "normal" #

    predictions.append(pred_label)
    ground_truth.append(true_label)
    texts.append(text)

# Create a DataFrame for results
results_df = pd.DataFrame({
    "text": texts,
    "ground_truth": ground_truth,
    "prediction": predictions
})

results_df.head()

Running inference...


100%|██████████| 100/100 [00:12<00:00,  8.08it/s]


,text,ground_truth,prediction
0,<user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children,normal,hatespeech
1,<user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass,normal,hatespeech
2,<user> <user> well she ’ muslim so of course she hates the jews jew hatred is one of the defining feature of islam,hatespeech,hatespeech
3,an these nigger biches look like godzilla nasty,hatespeech,hatespeech
4,education or marxist indoctrination since it is well established blacks are incapable of learning anything past the age of ten unless they are half white should not blacks go to a trade school upon age <number> say learning how to pick cotton fruits and vegetables,hatespeech,hatespeech


In [68]:
# Display full results with complete text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Complete Results - Full Text View:\n")
print(results_df.to_string(index=True))

Complete Results - Full Text View:

                                                                                                                                                                                                                                                                                                       text ground_truth  prediction
0                                                                                                                     <user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children       normal  hatespeech
1                                                                                                                                                                                                  <user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass       normal  

In [69]:
# Evaluation
print("\nClassification Report:")
report = classification_report(ground_truth, predictions, target_names=["hatespeech", "normal"], labels=["hatespeech", "normal"], output_dict=True)
report_df = pd.DataFrame(report).transpose()
display(report_df)

print("\nConfusion Matrix:")
cm = confusion_matrix(ground_truth, predictions, labels=["hatespeech", "normal"])
cm_df = pd.DataFrame(cm, index=["Actual Hatespeech", "Actual Normal"], columns=["Predicted Hatespeech", "Predicted Normal"])
display(cm_df)


Classification Report:


,precision,recall,f1-score,support
hatespeech,0.468085,1.000000,0.637681,44.0
normal,1.000000,0.107143,0.193548,56.0
accuracy,0.500000,0.500000,0.500000,0.5
macro avg,0.734043,0.553571,0.415615,100.0
weighted avg,0.765957,0.500000,0.388967,100.0



Confusion Matrix:


,Predicted Hatespeech,Predicted Normal
Actual Hatespeech,44,0
Actual Normal,50,6
